In [1]:
%pip install spacy --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
!python -m spacy download es_core_news_lg --quiet

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')


In [7]:
!pip install faiss-cpu --quiet

In [1]:
import pandas as pd
import spacy
import numpy as np
import re
import os
import time
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import faiss
import pprint

# Cargar corpus

In [2]:
file_path = r"C:\Users\karen\Documents\HumanidadesDigitales_git\BDD_Corpus\Corpus_completo_revisado.xlsx"
corpus_completo = pd.read_excel(file_path, engine='openpyxl')

In [3]:
# Formatear fechas

meses = {
    'enero': '01',
    'febrero': '02',
    'marzo': '03',
    'abril': '04',
    'mayo': '05',
    'junio': '06',
    'julio': '07',
    'agosto': '08',
    'septiembre': '09',
    'octubre': '10',
    'noviembre': '11',
    'diciembre': '12'
}

def convertir_fecha(fecha_str):
    partes = fecha_str.split(' de ')
    dia = partes[0].zfill(2)  # Asegura 2 dígitos para días 1-9
    mes = meses[partes[1]]
    año = partes[2]
    return f"{dia}/{mes}/{año}"  # Formato DD/MM/AAAA

corpus_completo['Fecha'] = corpus_completo['Fecha'].apply(convertir_fecha)
corpus_completo['Fecha'] = pd.to_datetime(corpus_completo['Fecha'], format='%d/%m/%Y', errors='coerce')

# Embeddings

In [4]:
# Generar embeddings de textos
modelo_embedding = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

In [5]:
textos = corpus_completo['Texto'].astype(str).tolist()  # Convertir a lista de strings

In [ ]:
#embeddings = modelo_embedding.encode(textos, convert_to_tensor=False).astype('float32')

In [ ]:
#np.save(r'Out/embeddingsIR.npy', embeddings)

In [6]:
embeddings = np.load(r'Out/embeddingsIR.npy')

In [8]:
# Crear índice FAISS
indice = faiss.IndexFlatL2(embeddings.shape[1])
indice.add(embeddings)

# Consulta

In [14]:
# Buscar texto similar a una consulta
consulta = ["Ciencia y tecnología"]

embedding_consulta = modelo_embedding.encode(consulta, convert_to_tensor=False).astype('float32')
distancias, indices = indice.search(embedding_consulta, k=1000)

In [10]:
#Resultados
print("Textos más similares:")
for idx in indices[0]:
    print(f"Índice: {idx}")
    pprint.pp("Texto: " + textos[idx])
    pprint.pp("---------------------------------------")

Textos más similares:
Índice: 10741
('Texto: El tema de ciencia y tecnología parece haber entrado a las campañas '
 'presidenciales. Todo el mundo está de acuerdo en que son importantes para el '
 'desarrollo del país. Eso debería alegrar; sin embargo, hay algo que empaña '
 'la felicidad: el poco aprecio por la ciencia básica (natural y social) y la '
 'insistencia en concentrarse solo en una ciencia ‘útil’.\n'
 '\n'
 'A primera vista parece razonable. ¿Para qué invertir esfuerzos y recursos en '
 'algo inútil? Solo que las cosas nunca han sido tan sencillas. El título de '
 'esta columna es tomado de un ensayo de Abraham Flexner (de 1939).\n'
 '\n'
 'Flexner fue uno de los fundadores del Instituto de Estudios Avanzados, en '
 'Princeton. Por ese instituto han pasado 34 premios nobel y 41 ganadores de '
 'la medalla Fields (el equivalente en matemáticas al Nobel). Algunos fueron '
 'Albert Einstein, Robert Oppenheimer, John von Neumann, Kurt Gödel, John Nash '
 'y una larga fila tras 

In [15]:
pprint.pp(textos[indices[0][999]])

('No sé qué fue peor, si la desaparición de los cursos de historia, efecto '
 'final de la agenda de la simplificación del pensamiento, o los efectos '
 'colaterales de haber perdido la oportunidad de desarrollar una disciplina '
 'crítica de la evolución cultural.\n'
 '\n'
 'Pese a que en mi infancia y juventud existía la “clase” de historia en el '
 'colegio, las cosas curiosamente se confabulaban para nunca superar en ellas '
 'el siglo 19 y, mucho menos, hablar de la Colombia contemporánea. La historia '
 'de los imperios y la formación de las naciones, la historia de los héroes e '
 'incluso la historia bíblica se repetían año tras año y el tiempo “no '
 'alcanzaba” en el programa para llegar a algo mínimamente relacionado con la '
 'propia vida. Me formé llena de huecos, un poco en las bibliotecas '
 'excepcionales de casa, un poco en la genealogía del hogar, cosas que '
 'implicaban mezclar un linaje aventurero franco suizo con uno de desplazados '
 'republicanos catalanes, tema

In [16]:
indices[0][999]

12499

In [20]:
corpus_completo.iloc[indices[0][999]]['Vínculo']

'https://web.archive.org/web/20180111225338/http://www.semana.com/opinion/articulo/clase-de-historia-en-los-colegios-opinion-de-brigitte-baptiste/552964'